# Mini Project 1 — Analysis Notebook

**Name:** Amy Cai  
**Dataset:** TMDB (movies + US streaming providers)  
**Date:** 5/20/2026

This notebook walks through Section 1 (overview), Section 2 (data profile), Section 3 (analysis), Section 3.5 (visualization), and Section 4 (conclusions). I use TMDB’s popular-movies API plus US watch-provider data to compare how titles spread across genres and subscription streaming services, and how user ratings relate to that spread.


In [1]:
# Setup — run this cell first
# Assignment requirement: keep this line (do not delete)
!pip install jupyter plotly kaleido pandas

# Extra packages this project uses (API + saving Plotly images)
!pip install python-dotenv requests nbformat

import pandas as pd
import plotly.express as px

print("Setup complete.")


zsh:1: command not found: pip
zsh:1: command not found: pip
Setup complete.


---

## Section 1 — Overview

### Dataset

This project uses movie data from **The Movie Database (TMDB)**. I pulled **50 pages** of titles from the [`/movie/popular`](https://developer.themoviedb.org/reference/movie-popular-list) endpoint and added US watch-provider data from [`/movie/{movie_id}/watch/providers`](https://developer.themoviedb.org/reference/movie-watch-providers) (subscription, rent, and buy options). Key fields include `title`, `genre_ids`, `vote_average`, `vote_count`, `popularity`, `release_date`, and `streaming_provider_names`. API docs: [developer.themoviedb.org/docs](https://developer.themoviedb.org/docs). Browse titles: [themoviedb.org](https://www.themoviedb.org/).

### Why this dataset

As an aspiring UX researcher interested in entertainment, I wanted live industry data—not a static CSV. Streaming apps shape what people discover and watch, so I asked how **popular** movies are distributed by **genre** and **platform**, and whether **user ratings** line up with where titles actually show up.

### Three analytical questions

1. How many distinct popular-movie titles fall in each genre on each US subscription streaming service, and where do those titles concentrate?
2. Within each streaming service, how is the sampled catalog split by genre?
3. On each streaming service, which genre has the highest mean TMDB user rating (`vote_average`)?

### What a practitioner would do with these findings

Product and content teams (or researchers comparing platforms) could use this to spot genre gaps, understand catalog mix, and flag high-rated niches worth promoting—while remembering the sample is TMDB “popular” titles only, not a full catalog audit.


---

## Section 2 — Data Profile

In the code cell below, I load data from the TMDB API, merge US watch-provider fields, and run basic quality checks. The markdown cell after it summarizes row counts, column meanings, issues, and which fields drive Sections 3 and 3.5.


In [14]:
# Load dataset from TMDB API (credentials in this folder's .env file)

import os
from pathlib import Path

import requests
from dotenv import load_dotenv

# Find the MP1 folder whether the notebook runs from MP1/ or the repo root
def _mp1_dir() -> Path:
    cwd = Path(".").resolve()
    if (cwd / "week6_mp1_starter.ipynb").is_file() and (cwd / ".env").is_file():
        return cwd
    if (cwd / "MP1" / "week6_mp1_starter.ipynb").is_file():
        return cwd / "MP1"
    return cwd

MP1_DIR = _mp1_dir()
env_path = MP1_DIR / ".env"
if env_path.is_file():
    load_dotenv(dotenv_path=env_path)
else:
    load_dotenv()

# Read the API key from .env or your shell environment (never paste keys in the notebook)
api_key = os.getenv("TMDB_API_KEY")
if not api_key:
    raise ValueError(
        "TMDB_API_KEY not found. Copy .env.example to .env in the MP1 folder "
        "and set your key, or export TMDB_API_KEY in your environment."
    )

# Pull many pages of /movie/popular (20 titles per page) so charts are not tiny-sample artifacts.
POPULAR_PAGES = 50
url = "https://api.themoviedb.org/3/movie/popular"
base_params = {"api_key": api_key, "language": "en-US"}

# Empty list that will hold movie records from every page
results = []
# Loop through page numbers 1 up to POPULAR_PAGES
for page in range(1, POPULAR_PAGES + 1):
    # Ask TMDB for one page of popular movies
    response = requests.get(url, params={**base_params, "page": page}, timeout=30)
    response.raise_for_status()  # stop if the request failed
    page_results = response.json().get("results", [])
    if not page_results:
        print(f"  Page {page} returned no results; stopping early.")
        break
    # Add this page's movies to the big list
    results.extend(page_results)
    if page % 10 == 0:
        print(f"  Fetched popular pages 1–{page} ({len(results):,} raw rows so far)")

# Turn the list of movie dicts into a pandas DataFrame (table)
df = pd.DataFrame(results)
# Same title should not appear twice across pages; keep first occurrence if it does.
if "id" in df.columns:
    df = df.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)

# Optional enrichment: add where each movie can be streamed/rented/bought (US region).
def get_watch_providers(movie_id, region="US"):
    # TMDB endpoint for one movie's watch options
    provider_url = f"https://api.themoviedb.org/3/movie/{movie_id}/watch/providers"
    provider_resp = requests.get(provider_url, params={"api_key": api_key}, timeout=30)
    provider_resp.raise_for_status()

    # Data for the country we care about (US)
    region_data = provider_resp.json().get("results", {}).get(region, {})
    flatrate = region_data.get("flatrate", [])  # subscription streaming
    rent = region_data.get("rent", [])
    buy = region_data.get("buy", [])

    # Build comma-separated provider name strings for this movie
    return {
        "streaming_provider_names": ", ".join(sorted({p.get("provider_name", "") for p in flatrate if p.get("provider_name")})),
        "rent_provider_names": ", ".join(sorted({p.get("provider_name", "") for p in rent if p.get("provider_name")})),
        "buy_provider_names": ", ".join(sorted({p.get("provider_name", "") for p in buy if p.get("provider_name")})),
        "watch_provider_link": region_data.get("link"),
    }

# One row of provider info per movie; we fill this in a loop
provider_rows = []
movie_ids = list(df["id"])
n_movies = len(movie_ids)
for i, movie_id in enumerate(movie_ids, start=1):
    try:
        provider_rows.append(get_watch_providers(movie_id, region="US"))
    except requests.RequestException:
        # If one movie fails, use blanks so the loop can keep going
        provider_rows.append(
            {
                "streaming_provider_names": "",
                "rent_provider_names": "",
                "buy_provider_names": "",
                "watch_provider_link": None,
            }
        )
    if i % 100 == 0 or i == n_movies:
        print(f"  Watch providers: {i:,} / {n_movies:,}")

# Turn provider rows into a table and glue it onto the right side of df
providers_df = pd.DataFrame(provider_rows)
df = pd.concat([df.reset_index(drop=True), providers_df], axis=1)

from IPython.display import display

print(df.shape)
display(df.head())

# --- Section 2 helpers: rows/columns, column meanings, quality checks, analysis focus ---

n_rows, n_cols = df.shape
print(f"\nRows: {n_rows:,}  |  Columns: {n_cols}")

# Plain-English descriptions of TMDB columns (for your write-up)
TMDB_POPULAR_FIELD_MEANINGS = {
    "adult": "Whether TMDB marks the title as adult content.",
    "backdrop_path": "CDN path for the backdrop image (often null).",
    "genre_ids": "List of numeric genre ids for this movie.",
    "id": "Unique TMDB movie id.",
    "original_language": "ISO 639-1 language code for the original language.",
    "original_title": "Title in the original language.",
    "overview": "Short plot summary text.",
    "popularity": "TMDB popularity score (algorithmic; not the same as vote count).",
    "poster_path": "CDN path for the poster image (often null).",
    "release_date": "Theatrical release date as YYYY-MM-DD string from the API.",
    "title": "Localized title for the requested language.",
    "video": "Whether TMDB treats this as a video entry (often False for features).",
    "vote_average": "Mean user rating from 0–10.",
    "vote_count": "Number of user ratings averaged into vote_average.",
    "streaming_provider_names": "Comma-separated subscription streaming services available in the selected region (US).",
    "rent_provider_names": "Comma-separated services where the movie can be rented in the selected region (US).",
    "buy_provider_names": "Comma-separated services where the movie can be purchased in the selected region (US).",
    "watch_provider_link": "TMDB link to the title's watch options page for the selected region (US).",
}

print("\n--- What each column represents (TMDB /movie/popular `results[]`) ---")
for col in df.columns:
    desc = TMDB_POPULAR_FIELD_MEANINGS.get(
        col,
        "(Not in the built-in cheat sheet — see TMDB response fields for this key.)",
    )
    print(f"  {col}: {desc}")

print("\n--- Missing values (NaN/NA in cells) ---")
missing = df.isna().sum()
if missing.any():
    print(missing[missing > 0].to_string())
else:
    print("No missing values detected at the cell level.")

if "id" in df.columns:
    print(f"\nDuplicate `id` values: {int(df['id'].duplicated().sum())}")

if "genre_ids" in df.columns:
    empty_genres = df["genre_ids"].apply(lambda x: isinstance(x, list) and len(x) == 0).sum()
    print(f"Rows with empty genre_ids list: {int(empty_genres)}")

if "release_date" in df.columns:
    parsed = pd.to_datetime(df["release_date"], errors="coerce")
    bad_release = df["release_date"].notna() & parsed.isna()
    n_bad = int(bad_release.sum())
    print(f"release_date strings that do not parse as dates: {n_bad}")
    if n_bad:
        print(df.loc[bad_release, "release_date"].head(10).to_string())

print("\n--- Dtypes (unexpected types / objects) ---")
print(df.dtypes.to_string())

print("\n--- Columns used in Sections 3 and 3.5 (see Data Profile markdown) ---")
focus = [
    c
    for c in [
        "id",
        "genre_ids",
        "streaming_provider_names",
        "vote_average",
    ]
    if c in df.columns
]
print("Columns:", focus)
print(
    "Why: id counts distinct movies after reshaping; genre_ids and "
    "streaming_provider_names drive genre × service counts and mix; "
    "vote_average answers highest mean rating per service. "
    "Sections 3 and 3.5 use subscription streaming only (646 of 985 titles)."
)


  Fetched popular pages 1–10 (200 raw rows so far)
  Fetched popular pages 1–20 (400 raw rows so far)
  Fetched popular pages 1–30 (600 raw rows so far)
  Fetched popular pages 1–40 (800 raw rows so far)
  Fetched popular pages 1–50 (1,000 raw rows so far)
  Watch providers: 100 / 985
  Watch providers: 200 / 985
  Watch providers: 300 / 985
  Watch providers: 400 / 985
  Watch providers: 500 / 985
  Watch providers: 600 / 985
  Watch providers: 700 / 985
  Watch providers: 800 / 985
  Watch providers: 900 / 985
  Watch providers: 985 / 985
(985, 19)


,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count,streaming_provider_names,rent_provider_names,buy_provider_names,watch_provider_link
0,False,/cccXiN28CPjcMVhji5LnogF4Njp.jpg,"[28, 18, 80]",1439930,The Punisher: One Last Kill,en,The Punisher: One Last Kill,As Frank Castle searches for meaning beyond re...,631.4432,/qQclTgLMDvGBuUBFGHRipxkEwWR.jpg,2026-05-12,False,False,8.525,947,Disney Plus,,,https://www.themoviedb.org/movie/1439930-the-p...
1,False,/2I1OFQJ0L9T0dpU6FobKFWV2PxX.jpg,"[878, 12]",687163,Project Hail Mary,en,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,417.1089,/yihdXomYb5kTeSivtFndMy5iDmf.jpg,2026-03-15,False,False,8.601,3658,,"Amazon Video, Apple TV Store, Fandango At Home...","Amazon Video, Apple TV Store",https://www.themoviedb.org/movie/687163-projec...
2,False,/zMwhWailP1WY7sb6AoE6b8ugoy.jpg,"[16, 10751, 12, 14]",1007757,Swapped,en,Swapped,"A small woodland creature and a majestic bird,...",325.0364,/tHhxWxge06goXU6ZQH1hj7vK8Hd.jpg,2026-05-01,False,False,8.986,1196,"Netflix, Netflix Standard with Ads",,,https://www.themoviedb.org/movie/1007757-swapp...
3,False,/9Z2uDYXqJrlmePznQQJhL6d92Rq.jpg,"[10751, 35, 12, 14, 16]",1226863,The Super Mario Galaxy Movie,en,The Super Mario Galaxy Movie,Having thwarted Bowser's previous plot to marr...,332.9093,/eJGWx219ZcEMVQJhAgMiqo8tYY.jpg,2026-04-01,False,False,7.125,1064,,"Amazon Video, Google Play Movies, YouTube","Google Play Movies, YouTube",https://www.themoviedb.org/movie/1226863-the-s...
4,False,/wMrV8SLne1jHLeYS0lLrA1Tf86P.jpg,"[27, 9648]",1304313,Lee Cronin's The Mummy,en,Lee Cronin's The Mummy,The young daughter of a journalist disappears ...,303.3424,/1q308iixueCU4pFtSYugNOevtNx.jpg,2026-04-15,False,False,7.682,454,,Amazon Video,,https://www.themoviedb.org/movie/1304313-lee-c...



Rows: 985  |  Columns: 19

--- What each column represents (TMDB /movie/popular `results[]`) ---
  adult: Whether TMDB marks the title as adult content.
  backdrop_path: CDN path for the backdrop image (often null).
  genre_ids: List of numeric genre ids for this movie.
  id: Unique TMDB movie id.
  title: Localized title for the requested language.
  original_language: ISO 639-1 language code for the original language.
  original_title: Title in the original language.
  overview: Short plot summary text.
  popularity: TMDB popularity score (algorithmic; not the same as vote count).
  poster_path: CDN path for the poster image (often null).
  release_date: Theatrical release date as YYYY-MM-DD string from the API.
  softcore: (Not in the built-in cheat sheet — see TMDB response fields for this key.)
  video: Whether TMDB treats this as a video entry (often False for features).
  vote_average: Mean user rating from 0–10.
  vote_count: Number of user ratings averaged into vote_average.


In [15]:
# Check column types and missing values
# .info() prints each column name, its data type, and how many non-null values it has
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 985 entries, 0 to 984
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   adult                     985 non-null    bool   
 1   backdrop_path             973 non-null    object 
 2   genre_ids                 985 non-null    object 
 3   id                        985 non-null    int64  
 4   title                     985 non-null    object 
 5   original_language         985 non-null    object 
 6   original_title            985 non-null    object 
 7   overview                  985 non-null    object 
 8   popularity                985 non-null    float64
 9   poster_path               985 non-null    object 
 10  release_date              985 non-null    object 
 11  softcore                  985 non-null    bool   
 12  video                     985 non-null    bool   
 13  vote_average              985 non-null    float64
 14  vote_count

In [16]:
# Summary statistics for numeric columns
# .describe() shows count, mean, min, max, etc. for number columns
df.describe()


,id,popularity,vote_average,vote_count
count,9.850000e+02,985.000000,985.000000,985.000000
mean,5.188851e+05,22.796753,6.861449,6525.331980
std,5.310419e+05,36.344918,1.440206,7273.815563
min,1.200000e+01,6.516200,0.000000,0.000000
25%,1.142300e+04,11.280900,6.500000,355.000000
50%,3.388030e+05,14.251800,7.111000,3870.000000
75%,1.003596e+06,20.792600,7.638000,10654.000000
max,1.691129e+06,631.443200,9.250000,39738.000000


### Data profile summary

#### How many rows and columns?

After fetching 50 pages of `/movie/popular` and deduplicating on `id`, the dataset has **985 rows** and **19 columns** (`df.shape` → `(985, 19)`).

#### What does each column represent?

Most fields come from TMDB’s popular-movies list: identifiers and text (`id`, `title`, `original_title`, `overview`), media paths (`poster_path`, `backdrop_path`), metadata (`adult`, `video`, `original_language`, `release_date`), and engagement metrics (`popularity`, `vote_average`, `vote_count`). `genre_ids` is a **list** of numeric genre codes per movie. I added US watch options from `/movie/{id}/watch/providers`: `streaming_provider_names` (subscription), `rent_provider_names`, `buy_provider_names`, and `watch_provider_link`.

#### Data quality issues

- **Missing values:** `backdrop_path` (12 missing), `watch_provider_link` (125 missing).
- **Duplicates:** **0** duplicate `id` values.
- **Genres:** **3** rows have an empty `genre_ids` list.
- **Dates:** All `release_date` strings parsed successfully (**0** bad dates).

#### Columns used in analysis (and why)

- **`genre_ids`** (mapped to genre names) — genre counts and mix by service.
- **`streaming_provider_names`** — US **subscription** services only in Sections 3 and 3.5 (`rent_provider_names` and `buy_provider_names` are loaded but not used in aggregates).
- **`vote_average`** — highest mean rating per service and genre.
- **`id`** — count **distinct movies** after exploding genres and providers.

`title` and `popularity` help interpret examples but are not the main aggregates in Sections 3 and 3.5.

#### Sample size for Sections 3 and 3.5

Of **985** popular titles, **646** have at least one non-empty US **subscription** provider. Sections 3 and 3.5 filter to those 646 movies (titles with only rent/buy or no provider are excluded). That step is documented again in the Question 1 cleaning table.


---

## Section 3 — Analysis

Section 3 uses **pandas only** for tables and aggregates; charts are saved for Section 3.5. The three questions build on each other:

- **Question 1** — *Where* do titles concentrate (counts by genre × service)?
- **Question 2** — *What mix* does each service show as shares within its catalog?
- **Question 3** — *Which genre rates highest* per service, and is that the same as the biggest genre by count?

Each question has a code cell, a cleaning/reshaping note, and an interpretation. Analysis uses the **646** subscription titles described in Section 2.


### Question 1

How many distinct popular-movie titles fall in each genre on each US subscription streaming service, and where do those titles concentrate?


In [17]:
# Question 1 — distinct title counts by (streaming service × genre)
# Reshape to long form: one row per movie × genre × US subscription service.

from IPython.display import display

# TMDB uses number codes for genres; this dict turns codes into readable names
MOVIE_GENRE_MAP = {
    28: "Action", 12: "Adventure", 16: "Animation", 35: "Comedy", 80: "Crime",
    99: "Documentary", 18: "Drama", 10751: "Family", 14: "Fantasy", 36: "History",
    27: "Horror", 10402: "Music", 9648: "Mystery", 10749: "Romance", 878: "Sci-Fi",
    10770: "TV Movie", 53: "Thriller", 10752: "War", 37: "Western",
}

# Start from a copy so we do not change the original df
long_df = df.copy()
# explode: if a movie has 3 genres, make 3 rows (one per genre)
long_df = long_df.explode("genre_ids")
# Replace genre id numbers with genre names; unknown ids become "Unknown"
long_df["genre"] = long_df["genre_ids"].map(MOVIE_GENRE_MAP).fillna("Unknown")
# Treat missing provider text as empty string so we can split safely
long_df["streaming_provider_names"] = (
    long_df["streaming_provider_names"].fillna("").astype(str)
)
# Split "Netflix, Hulu" into a list, then explode so each service gets its own row
long_df["streaming_provider_names"] = long_df["streaming_provider_names"].str.split(r",\s*")
long_df = long_df.explode("streaming_provider_names")
# Keep only rows that have a real subscription service name
long_df = long_df[long_df["streaming_provider_names"].str.strip() != ""]

# Count unique movies for each (service, genre) pair
genre_service_counts = (
    long_df.groupby(["streaming_provider_names", "genre"], as_index=False)
    .agg(movie_count=("id", "nunique"))
)

# Pivot: rows = genres, columns = services, values = movie counts
count_pivot = genre_service_counts.pivot_table(
    index="genre",
    columns="streaming_provider_names",
    values="movie_count",
    aggfunc="sum",
    fill_value=0,
)
# Sort genres so the busiest genres are at the top
count_pivot = count_pivot.loc[count_pivot.sum(axis=1).sort_values(ascending=False).index]
# Sort services so the busiest services are on the left
count_pivot = count_pivot[count_pivot.sum(axis=0).sort_values(ascending=False).index]

# Only keep the top 30 services so tables and charts stay readable
TOP_N_SERVICES = 30
n_services_in_pivot = count_pivot.shape[1]
if n_services_in_pivot > TOP_N_SERVICES:
    top_svc_cols = count_pivot.sum(axis=0).sort_values(ascending=False).head(TOP_N_SERVICES).index
    count_pivot = count_pivot[top_svc_cols]

# Make sure every genre row exists (use 0 if that genre has no movies on a service)
_all_genre_names = list(dict.fromkeys(MOVIE_GENRE_MAP.values()))
_full_genre_index = _all_genre_names + ["Unknown"]
_extra = sorted(set(genre_service_counts["genre"]) - set(_full_genre_index))
count_pivot = count_pivot.reindex(_full_genre_index + _extra, fill_value=0).fillna(0)
count_pivot = count_pivot.loc[count_pivot.sum(axis=1).sort_values(ascending=False).index]

# Quick summary numbers for the printout below
n_movies_with_streaming = long_df["id"].nunique()
n_service_genre_pairs = len(genre_service_counts)
print(f"Movies with ≥1 US subscription provider in sample: {n_movies_with_streaming:,}")
print(f"Distinct (service, genre) pairs: {n_service_genre_pairs:,}")
print(f"Services in analysis (top {count_pivot.shape[1]} by title volume): {count_pivot.shape[1]}")
print(f"Genres in pivot: {count_pivot.shape[0]}")

# Top 15 busiest service + genre combinations
busiest_cells = (
    genre_service_counts[genre_service_counts["streaming_provider_names"].isin(count_pivot.columns)]
    .sort_values("movie_count", ascending=False)
    .head(15)
    .rename(columns={"streaming_provider_names": "service"})
)
print("\nTop 15 (service, genre) pairs by distinct title count:")
display(busiest_cells)

# Total title-genre rows per service (a movie with 2 genres counts twice)
titles_per_service = (
    genre_service_counts.groupby("streaming_provider_names", as_index=False)["movie_count"]
    .sum()
    .rename(columns={"streaming_provider_names": "service", "movie_count": "titles_in_sample"})
    .sort_values("titles_in_sample", ascending=False)
)
print("\nTotal distinct title–genre rows per service (sums across genres; a movie can count in multiple genres):")
display(titles_per_service.head(10))


Movies with ≥1 US subscription provider in sample: 646
Distinct (service, genre) pairs: 705
Services in analysis (top 30 by title volume): 30
Genres in pivot: 20

Top 15 (service, genre) pairs by distinct title count:


,service,genre,movie_count
193,Disney Plus,Adventure,91
198,Disney Plus,Family,68
192,Disney Plus,Action,58
666,YouTube TV,Action,57
679,fuboTV,Action,56
194,Disney Plus,Animation,53
231,HBO Max,Action,53
246,HBO Max Amazon Channel,Action,52
232,HBO Max,Adventure,50
247,HBO Max Amazon Channel,Adventure,49



Total distinct title–genre rows per service (sums across genres; a movie can count in multiple genres):


,service,titles_in_sample
29,Disney Plus,433
41,HBO Max,320
42,HBO Max Amazon Channel,312
61,Netflix,283
94,fuboTV,280
62,Netflix Standard with Ads,264
93,YouTube TV,263
5,Amazon Prime Video,252
6,Amazon Prime Video with Ads,246
73,Philo,191


### Cleaning and reshaping (Question 1)

| Step | What changed | Why |
|------|----------------|-----|
| Start from `df` | One row per movie from Section 2 | Base table after API load and watch-provider merge |
| `explode("genre_ids")` | One row per movie × genre | Movies have multiple genres; counts need genre-level rows |
| `map(MOVIE_GENRE_MAP)` → `genre` | Numeric ids → genre names; unmapped → `"Unknown"` | Readable genres aligned with TMDB codes |
| `fillna("")` + `astype(str)` on `streaming_provider_names` | Missing providers → empty string | Safe to split provider lists |
| `str.split(r",\s*")` + `explode(...)` | One row per movie × genre × subscription service | A title on Netflix and Hulu becomes two rows |
| Filter `streaming_provider_names != ""` | Drops rent-only / buy-only / no-provider titles | **646** movies with US subscription streaming |
| `groupby(...).agg(movie_count=("id", "nunique"))` | Distinct movies per (service, genre) | `nunique` avoids double-counting after explode |
| `pivot_table` → `count_pivot` | Service × genre grid | See concentration in both dimensions |
| Sort rows/columns by total volume | Busiest genres and services first | Highlights where titles pile up |
| Keep top **30** services (`TOP_N_SERVICES`) | Drops long tail of sparse providers | Keeps tables and charts readable |
| `reindex` full genre list + `fill_value=0` | Every genre row present | Stable genre axis for comparisons |

**Objects for later questions:** `long_df`, `genre_service_counts`, `count_pivot`


### Interpretation (Question 1)

Among the **646** popular movies with at least one US subscription provider, title volume is **not evenly spread** across services or genres. **Disney Plus** has the largest footprint (433 title–genre rows summed across genres), well ahead of HBO Max (~320) and Netflix (~283). The busiest cells are **Disney Plus × Adventure** (91 distinct titles) and **Disney Plus × Family** (68)—in line with Disney’s brand and what I expected.

Beyond Disney, **Action** and **Adventure** recur in the top pairs (YouTube TV, fuboTV, HBO Max and Amazon Channel variants), so many platforms lean on broad, franchise-friendly genres. HBO Max and its channel twin track closely; TMDB lists separate provider names for bundles users may see as one subscription.

**Next:** raw counts do not show each service’s *share* by genre—that is Question 2.


### Question 2

Within each streaming service, how is the sampled catalog split by genre?


In [18]:
# Question 2 — within-service genre mix (% of each service's title–genre rows)
# Uses count_pivot from Question 1 (run that cell first).

# Use the same top 10 genres as in the cleaning notes
TOP_GENRES_FOR_MIX = 10
major_genres = (
    count_pivot.sum(axis=1).sort_values(ascending=False).head(TOP_GENRES_FOR_MIX).index.tolist()
)

# Build one small table per service, then stack them together
mix_parts = []
for svc in count_pivot.columns:
    col = count_pivot[svc]
    total = col.sum()
    if total == 0:
        continue
    # Counts for the 10 biggest genres
    major_counts = col.reindex(major_genres, fill_value=0)
    # Everything else goes into "Other"
    other_count = col.drop(index=major_genres, errors="ignore").sum()
    svc_mix = pd.DataFrame({
        "streaming_provider_names": svc,
        "genre": list(major_genres) + ["Other"],
        "title_genre_rows": list(major_counts.astype(int)) + [int(other_count)],
    })
    # Percent of this service's rows in each genre
    svc_mix["pct"] = 100.0 * svc_mix["title_genre_rows"] / total
    mix_parts.append(svc_mix)

# One long table with all services
mix_df = pd.concat(mix_parts, ignore_index=True)

# For each service, show its top 2 genres by percent
top2_mix = (
    mix_df[mix_df["genre"] != "Other"]
    .sort_values(["streaming_provider_names", "pct"], ascending=[True, False])
    .groupby("streaming_provider_names", as_index=False)
    .head(2)
)
print(f"Genre mix uses top {TOP_GENRES_FOR_MIX} genres by volume plus an Other bucket.")
print("Largest genre shares per service (top 2 genres):")
display(top2_mix)

# Compare genre mix on the 5 biggest services side by side
compare_services = count_pivot.sum(axis=0).sort_values(ascending=False).head(5).index.tolist()
mix_compare = mix_df[mix_df["streaming_provider_names"].isin(compare_services)].pivot_table(
    index="genre",
    columns="streaming_provider_names",
    values="pct",
    fill_value=0.0,
)
mix_compare = mix_compare.loc[mix_compare.sum(axis=1).sort_values(ascending=False).index]
print("\nPercent of each service's rows by genre (top 5 services by volume):")
display(mix_compare.round(1))


Genre mix uses top 10 genres by volume plus an Other bucket.
Largest genre shares per service (top 2 genres):


,streaming_provider_names,genre,title_genre_rows,pct
267,AMC Plus Apple TV Channel,Thriller,8,21.621622
273,AMC Plus Apple TV Channel,Horror,8,21.621622
289,AMC+,Thriller,7,22.580645
295,AMC+,Horror,7,22.580645
245,AMC+ Amazon Channel,Thriller,10,23.255814
244,AMC+ Amazon Channel,Drama,7,16.279070
77,Amazon Prime Video,Action,37,14.682540
79,Amazon Prime Video,Drama,37,14.682540
88,Amazon Prime Video with Ads,Action,36,14.634146
90,Amazon Prime Video with Ads,Drama,36,14.634146



Percent of each service's rows by genre (top 5 services by volume):


streaming_provider_names,Disney Plus,HBO Max,HBO Max Amazon Channel,Netflix,fuboTV
genre,,,,,
Action,13.4,16.6,16.7,12.0,20.0
Adventure,21.0,15.6,15.7,9.2,13.2
Other,16.9,11.6,11.5,19.8,12.5
Fantasy,9.7,14.7,14.7,4.2,3.6
Sci-Fi,9.9,8.4,8.3,8.1,9.3
Comedy,9.7,4.4,4.5,10.6,10.0
Drama,3.0,5.6,5.8,12.7,11.1
Thriller,0.2,9.4,9.0,10.6,8.2
Family,15.7,3.1,3.2,4.2,4.3


### Cleaning and reshaping (Question 2)

This question reuses **`count_pivot`** from Question 1 (no new API call).

| Step | What changed | Why |
|------|----------------|-----|
| Pick top **10** genres by total count (`TOP_GENRES_FOR_MIX`) | `major_genres` list | Focus on largest genres; bundle the rest as **Other** |
| For each service column in `count_pivot` | Loop over services | Mix is computed **within** each service |
| `title_genre_rows` per genre | Raw counts from the pivot | Numerator for share calculations |
| `pct = 100 × title_genre_rows / total` | Percent of that service’s rows | Composition, not raw volume |
| **Other** bucket | Sum of non–top-10 genres | Percentages sum to 100% |
| `pd.concat` → `mix_df` | Long table: service, genre, counts, % | Tables and Section 3.5 stacked-bar chart |
| `pivot_table` on top 5 services → `mix_compare` | Genre × service grid of **%** | Side-by-side mix for the busiest services |

**Note:** Percentages use **title–genre rows** (after exploding), so multi-genre movies contribute to more than one slice.


### Interpretation (Question 2)

Question 2 shifts from volume to **within-service composition**. **Disney Plus** is the clearest outlier: **Adventure (~21%)** and **Family (~16%)** dominate, while **Drama (~3%)** and **Thriller (~0.2%)** are tiny. **Netflix** looks more balanced: **Drama (~13%)**, **Action (~12%)**, and **Other (~20%)**.

Among the top five services by volume, **HBO Max** leans more on **Fantasy (~15%)**, **Thriller (~9%)**, and **Horror (~6–7%)** than Disney. **fuboTV** and **YouTube TV** peak on **Action (~20–22%)** plus **Adventure**. Niche services skew sharply (e.g., **Criterion Channel** ~30% Drama; AMC properties **Thriller/Horror** ~22% each).

**Next:** a large slice does not guarantee the highest ratings—that is Question 3.


### Question 3

On each streaming service, which genre has the highest mean TMDB user rating (`vote_average`)?


In [19]:
# Question 3 — highest mean-rated genre per service
# Uses long_df from Question 1 (run that cell first).

# Skip genre-service pairs with only 1–2 movies (averages would be shaky)
MIN_TITLES_FOR_GENRE = 3

# Mean and median rating plus movie count for each (service, genre)
avg_rating_by_service_genre = (
    long_df.groupby(["streaming_provider_names", "genre"], as_index=False)
    .agg(
        avg_vote_average=("vote_average", "mean"),
        movie_count=("id", "nunique"),
        median_vote_average=("vote_average", "median"),
    )
)

# Keep only pairs with enough movies
rated_genres = avg_rating_by_service_genre[
    avg_rating_by_service_genre["movie_count"] >= MIN_TITLES_FOR_GENRE
].copy()

# For each service, pick the genre with the highest mean rating (one winner per service)
top_genre_per_service = (
    rated_genres.sort_values(
        ["streaming_provider_names", "avg_vote_average", "movie_count"],
        ascending=[True, False, False],
    )
    .drop_duplicates(subset=["streaming_provider_names"], keep="first")
    .sort_values("avg_vote_average", ascending=False)
)

# Same service subset as Question 1 (top services by title volume).
top_genre_per_service = top_genre_per_service[
    top_genre_per_service["streaming_provider_names"].isin(count_pivot.columns)
].copy()

print(
    f"Winner per service = highest mean vote_average "
    f"(genres with <{MIN_TITLES_FOR_GENRE} titles excluded)."
)
print(f"Services compared: {len(top_genre_per_service)}")
display(
    top_genre_per_service[
        ["streaming_provider_names", "genre", "avg_vote_average", "movie_count", "median_vote_average"]
    ]
    .rename(columns={
        "streaming_provider_names": "service",
        "avg_vote_average": "mean_rating",
        "movie_count": "titles_in_genre",
        "median_vote_average": "median_rating",
    })
    .round({"mean_rating": 2, "median_rating": 2})
)

# Average rating across all rows in long_df (for comparison)
overall_mean = long_df["vote_average"].mean()
print(f"\nOverall mean vote_average in long-form sample: {overall_mean:.2f}")
above_overall = top_genre_per_service[
    top_genre_per_service["avg_vote_average"] > overall_mean
]
print(
    f"Services whose winning genre mean is above the sample overall ({overall_mean:.2f}): "
    f"{len(above_overall)} / {len(top_genre_per_service)}"
)

# How many services each genre "won" on
winning_genre_counts = (
    top_genre_per_service["genre"].value_counts().rename_axis("genre").reset_index(name="services_won")
)
print("\nHow often each genre 'wins' highest mean rating across services:")
display(winning_genre_counts)


Winner per service = highest mean vote_average (genres with <3 titles excluded).
Services compared: 30


,service,genre,mean_rating,titles_in_genre,median_rating
533,Philo,Western,8.31,3,8.27
322,MGM Plus,Western,8.23,3,8.27
695,fuboTV,Western,8.22,4,8.23
449,Paramount Plus Premium,Western,8.14,3,8.19
388,Netflix,History,8.10,5,8.19
406,Netflix Standard with Ads,History,8.10,5,8.19
200,Disney Plus,Music,8.01,3,8.09
674,YouTube TV,Mystery,7.89,3,7.89
506,Peacock Premium Plus,Mystery,7.86,3,8.18
491,Peacock Premium,Mystery,7.86,3,8.18



Overall mean vote_average in long-form sample: 7.22
Services whose winning genre mean is above the sample overall (7.22): 30 / 30

How often each genre 'wins' highest mean rating across services:


,genre,services_won
0,Drama,7
1,Animation,6
2,Western,4
3,Mystery,3
4,History,2
5,Action,2
6,Comedy,2
7,Music,1
8,Romance,1
9,Family,1


### Cleaning and reshaping (Question 3)

This question reuses **`long_df`** and the same **top 30** services as **`count_pivot`** from Question 1.

| Step | What changed | Why |
|------|----------------|-----|
| `groupby` service + genre on `long_df` | Mean/median `vote_average`, `nunique` movie count | Rating aggregates per (service, genre) |
| Filter `movie_count >= 3` (`MIN_TITLES_FOR_GENRE`) | Drops pairs with 1–2 titles | Less shaky means |
| Sort by mean (then count), `drop_duplicates` per service | One **winning** genre per service | Highest mean `vote_average` per platform |
| Filter winners to `count_pivot.columns` | Same **top 30** services as Q1 | Same service set as counts and mix |
| `value_counts` on winning genres | How often each genre “wins” | Pattern across platforms |

**Object for Section 3.5:** `top_genre_per_service`


### Interpretation (Question 3)

For each of the **30** busiest services, the winning genre has the **highest mean `vote_average`**, with at least three titles in that pair. Top means cluster around **7.4–8.3** (overall sample mean ≈ **7.22**), so each service’s best genre beats the pooled average by design.

Winners are **not** the same as the biggest genres by count from Question 1. **Drama** wins on seven services; **Western** has the highest individual means but often only **3–4** titles. **Animation** wins on six services, including HBO Max (**18** titles, mean 7.82)—a stabler signal. Netflix’s winner is **History** (8.10, five titles), not its largest Action/Drama buckets.

Highly rated niches can outscore bulk genres even when they are a small catalog slice. I would not treat “winning genre” as “what subscribers watch most.”


---

## Section 3.5 — Visualization

The code cell below builds **three Plotly charts**, one per research question. Chart 2 uses stacked bars because genre *mix* is easier to read as shares than as a second heatmap.

| Chart | Question | Chart type |
|-------|----------|------------|
| 1 | Title counts by genre × service | Heatmap |
| 2 | Within-service genre mix | 100% stacked horizontal bars |
| 3 | Highest mean-rated genre per service | Horizontal bar chart |

Run Section 3 first so `count_pivot`, `mix_df`, and `top_genre_per_service` exist. Rationale and takeaways for each chart follow the code cell.


In [20]:
# Section 3.5 — Visualization only (run Section 3 Questions 1–3 first)
# Uses: count_pivot, mix_df, top_genre_per_service, long_df

from pathlib import Path

# Make sure Section 3 has been run — these variables must already exist
_required = ["count_pivot", "mix_df", "top_genre_per_service", "long_df"]
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise NameError(
        f"Run Section 3 analysis cells first; missing: {', '.join(_missing)}"
    )

# Figure out which folder to save PNG files into (depends where you run the notebook)
_cwd = Path(".").resolve()
if (_cwd / "week6_mp1_starter.ipynb").is_file():
    CHART_OUT_DIR = _cwd
elif (_cwd / "MP1" / "week6_mp1_starter.ipynb").is_file():
    CHART_OUT_DIR = _cwd / "MP1"
else:
    CHART_OUT_DIR = _cwd


def _save_chart(fig, filename: str, **kwargs):
    path = CHART_OUT_DIR / filename
    try:
        fig.write_image(str(path), **kwargs)
        print(f"Saved chart: {path}")
    except Exception as exc:
        print(f"Could not save {filename!r}: {exc}")


# --- Chart 1 (Question 1): title counts by service × genre ---
n_movies_sample = long_df["id"].nunique()
max_cell = int(count_pivot.to_numpy().max())
n_genres = len(count_pivot.index)

# Heatmap: darker color = more movies in that genre + service cell
fig = px.imshow(
    count_pivot,
    labels={
        "x": "Streaming service (US, subscription)",
        "y": "Genre",
        "color": "Distinct movie titles",
    },
    color_continuous_scale="Blues",
    aspect="auto",
)
fig.update_xaxes(side="bottom", tickangle=-45, automargin=True, title=dict(standoff=14))
fig.update_yaxes(ticklabelstep=1, automargin=True, tickfont=dict(size=10), title=dict(standoff=10))
fig.update_layout(
    title=dict(
        text="Movie-title counts by genre and streaming service (TMDB popular, US)",
        subtitle=dict(
            text=(
                f"{n_movies_sample:,} movies, {n_genres} genre rows, "
                f"top {count_pivot.shape[1]} services by title volume; "
                f"busiest cell has {max_cell} titles."
            ),
            font=dict(size=13, color="#444"),
        ),
        font=dict(size=18),
        x=0.02,
        xanchor="left",
    ),
    height=max(620, 22 * n_genres),
    width=1000,
    margin=dict(l=16, r=110, t=150, b=160),
    paper_bgcolor="white",
    coloraxis_colorbar=dict(
        title=dict(text="Distinct<br>movie titles", side="right"),
        thickness=20,
        len=0.82,
        outlinewidth=0,
        y=0.5,
        yanchor="middle",
    ),
)
fig.show()
_heat_h = max(620, 22 * n_genres)
_save_chart(fig, "mp1_genre_service_heatmap.png", width=1000, height=_heat_h, scale=2)

# --- Chart 2 (Question 2): within-service genre mix ---
TOP_GENRES_FOR_STACK = TOP_GENRES_FOR_MIX
genre_stack_order = [g for g in major_genres] + ["Other"]
mix_plot = mix_df.copy()
# Fix genre order so stacked bars always use the same color order
mix_plot["genre"] = pd.Categorical(mix_plot["genre"], categories=genre_stack_order, ordered=True)
# Order services from smallest bar to largest (bottom to top on chart)
svc_bar_order = count_pivot.sum(axis=0).sort_values(ascending=True).index.tolist()
mix_plot["streaming_provider_names"] = pd.Categorical(
    mix_plot["streaming_provider_names"], categories=svc_bar_order, ordered=True
)
mix_plot = mix_plot.sort_values(["streaming_provider_names", "genre"])

# Horizontal stacked bars: each bar is one service, slices are genre %
fig_mix = px.bar(
    mix_plot,
    x="pct",
    y="streaming_provider_names",
    color="genre",
    orientation="h",
    barmode="stack",
    labels={
        "streaming_provider_names": "Streaming service",
        "pct": "Share of this service's titles (%)",
        "genre": "Genre",
    },
)
fig_mix.update_layout(
    title=dict(
        text="Within each service, how are sampled titles split by genre?",
        subtitle=dict(
            text=(
                f"100% stacked bars: top {TOP_GENRES_FOR_STACK} genres by volume in your sample, "
                "plus Other."
            ),
            font=dict(size=13, color="#444"),
        ),
        font=dict(size=18),
        x=0.02,
        xanchor="left",
    ),
    xaxis=dict(
        title=dict(text="Share of this service's titles (%)", standoff=12),
        range=[0, 100],
        dtick=20,
        automargin=True,
    ),
    yaxis=dict(
        title=dict(text="Streaming service", standoff=8),
        automargin=True,
        tickfont=dict(size=11),
    ),
    height=max(580, 24 * len(svc_bar_order)),
    width=1000,
    margin=dict(l=20, r=200, t=150, b=72),
    paper_bgcolor="white",
    legend=dict(
        title=dict(text="Genre"),
        orientation="v",
        yanchor="top",
        y=1,
        x=1.01,
        xanchor="left",
        font=dict(size=9),
    ),
)
fig_mix.show()
_mix_h = max(580, 24 * len(svc_bar_order))
_save_chart(fig_mix, "mp1_genre_mix_within_service.png", width=1000, height=_mix_h, scale=2)

# --- Chart 3 (Question 3): highest mean-rated genre per service ---
top_for_plot = top_genre_per_service.copy()
# Text labels on bars (one decimal place)
top_for_plot["rating_lbl"] = top_for_plot["avg_vote_average"].map(lambda v: f"{v:.1f}")
top_for_plot = top_for_plot.sort_values("avg_vote_average", ascending=True)
n_bars = len(top_for_plot)

# One bar per service: length = mean rating, color = winning genre
fig_top = px.bar(
    top_for_plot,
    x="avg_vote_average",
    y="streaming_provider_names",
    color="genre",
    text="rating_lbl",
    orientation="h",
    labels={
        "streaming_provider_names": "Streaming service",
        "avg_vote_average": "Mean TMDB rating (0–10)",
        "genre": "Winning genre",
    },
    hover_data={
        "genre": True,
        "movie_count": True,
        "avg_vote_average": ":.2f",
    },
)
fig_top.update_traces(textposition="outside", cliponaxis=False, textfont_size=11)
fig_top.update_layout(
    title=dict(
        text="Which genre has the highest mean TMDB rating on each service?",
        subtitle=dict(
            text=(
                "One winner per service (highest mean vote_average). "
                f"Services: top {len(count_pivot.columns)} by title volume in your sample."
            ),
            font=dict(size=13, color="#444"),
        ),
        font=dict(size=18),
        x=0.02,
        xanchor="left",
    ),
    xaxis=dict(
        title=dict(text="Mean TMDB user rating (0–10 scale)", standoff=12),
        range=[0, 11.2],
        automargin=True,
    ),
    yaxis=dict(
        title=dict(text="Streaming service", standoff=10),
        automargin=True,
        tickfont=dict(size=11),
    ),
    height=max(800, 42 * n_bars),
    width=1050,
    margin=dict(l=18, r=210, t=170, b=72),
    paper_bgcolor="white",
    legend=dict(
        title=dict(text="Winning genre"),
        orientation="v",
        yanchor="top",
        y=1,
        x=1.01,
        xanchor="left",
        font=dict(size=9),
    ),
)
fig_top.show()
_top_h = max(800, 42 * n_bars)
_save_chart(fig_top, "mp1_top_genre_mean_rating.png", width=1050, height=_top_h, scale=2)


Saved chart: /Users/amy/Code/hcde530/Week 6/mp1_genre_service_heatmap.png


Saved chart: /Users/amy/Code/hcde530/Week 6/mp1_genre_mix_within_service.png


Saved chart: /Users/amy/Code/hcde530/Week 6/mp1_top_genre_mean_rating.png


### Chart 1 — Movie title counts by genre and streaming service

#### Chart rationale

A heatmap fits two categorical variables (genre and service) and one numeric outcome (distinct title count per cell). Color intensity shows dense vs. sparse cells without stacking dozens of genres per bar. Sorting by volume highlights where titles concentrate (Question 1).

#### Takeaway

Disney Plus and Adventure/Family cells are the densest; Action and Adventure recur on several large services, so volume clusters by brand as much as by genre.


### Chart 2 — Within each service, how are sample titles split by genre?

#### Chart rationale

100% stacked horizontal bars compare **composition** within each service. Every bar sums to 100%, so mix is comparable even when total title counts differ. This supplements the heatmap (Question 2).

#### Takeaway

Services do not share one default mix: Disney Plus is Adventure/Family-heavy, Netflix is more balanced, HBO leans Fantasy/Thriller/Horror.


### Chart 3 — Which genre has the highest TMDB rating on each service?

#### Chart rationale

A horizontal bar chart compares one number per service (mean rating of the winning genre). The vertical axis fits long provider names; color shows which genre won; hover shows title count, which matters when means rest on few movies (Question 3).

#### Takeaway

The top-rated genre per service is often a smaller niche (Western, History, Mystery)—not the genres with the most titles on the heatmap.


---

## Section 4 — Conclusions

This section summarizes findings across all three questions, notes limitations, and points to competency claims in `mp1.md`.


### Summary of findings

The main result is that **volume and quality tell different stories**. Action and Adventure dominate where titles pile up (Questions 1–2), but the highest mean TMDB ratings per service often come from smaller genres—Western, History, Music, Mystery, and Drama—not from the busiest heatmap cells (Question 3).

I was surprised that every top-30 service’s winning genre had a mean above the overall sample mean (**7.22**), even when that winner rested on only a handful of titles. That suggests the rating comparison is sensitive to small samples and genre choice.

### What I would investigate next

I would pull more titles per service, require more than three titles per genre before comparing means, and optionally weight ratings by `vote_count`. I would also group TMDB “channel” variants (e.g., HBO Max vs. HBO Max Amazon Channel) into parent brands.

### Limitations

- **Sample:** TMDB “popular” list (~985 titles; **646** with US subscription)—not full Netflix/HBO/etc. catalogs.
- **Ratings:** TMDB user scores, not proof one service is objectively better.
- **Providers:** Separate TMDB names for the same brand can split counts across columns.
- **Methods:** Top 30 services, top 10 genres plus Other, minimum 3 titles per genre for rating winners.

### Competency claims

See **`mp1.md`** in this repository for competency claims (C3, C5, C6, C7).
